In [7]:
!pip install fastapi uvicorn pyngrok transformers torch python-multipart -q

In [ ]:
from fastapi import FastAPI, UploadFile, File
from transformers import pipeline
import tempfile
import shutil

app = FastAPI()

print("Loading model...")

classifier = pipeline(
    "audio-classification",
    model="MelodyMachine/Deepfake-audio-detection-V2",
    device=0
)

print("Model loaded successfully!")

@app.get("/")
def home():
    return {"message": "Deepfake Audio Detection API Running"}

@app.post("/predict")
async def predict(file: UploadFile = File(...)):

    try:

        with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as temp:

            shutil.copyfileobj(file.file, temp)

            temp_path = temp.name

        result = classifier(temp_path)

        top = result[0]

        return {
            "label": top["label"],
            "confidence": round(top["score"] * 100, 4)
        }

    except Exception as e:

        return {
            "error": str(e)
        }

Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
import threading
import uvicorn

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run)
thread.start()

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3DlB7MkbFfw3iU6B0F8bwPNepF0_5hxnk76mbEpyyukDGkhBe")

In [8]:
public_url = ngrok.connect(8000)

print(public_url)

NgrokTunnel: "https://slip-viable-empathy.ngrok-free.dev" -> "http://localhost:8000"


In [ ]:
!pkill -f uvicorn